Upload the files from the Links provided in the comment and add it into the content section

In [4]:
import pandas as pd

# Path to your file in Colab
file_path = '/content/Suburbs and Localities, SA1 Distributions, SEIFA 2021.xlsx'#https://www.abs.gov.au/statistics/people/people-and-communities/socio-economic-indexes-areas-seifa-australia/2021#data-downloads

# The 4 tables (sheets)
sheets = ['Table 1', 'Table 2', 'Table 3', 'Table 4']

# Output filenames
output_names = {
    'Table 1': 'SAL_Decile_Disadvantage.csv',
    'Table 2': 'SAL_Decile_Adv_Disadv.csv',
    'Table 3': 'SAL_Decile_Economic_Resources.csv',
    'Table 4': 'SAL_Decile_Employment.csv'
}

def extract_simple_tables(path):
    for sheet in sheets:
        try:
            # 1. Load the sheet, skipping the ABS header rows
            # We use usecols=[0, 1, 2] to only grab the first 3 columns
            df = pd.read_excel(path, sheet_name=sheet, skiprows=5, usecols=[0, 1, 2])

            # 2. Rename columns for clarity based on your requirement
            df.columns = ['SAL Code', 'SAL Name', 'Decile Rank']

            # 3. Clean: Remove empty rows (footer notes)
            df = df.dropna(subset=['SAL Code'])

            # 4. Ensure SAL Code keeps its format (as a string)
            df['SAL Code'] = df['SAL Code'].astype(str).str.split('.').str[0]

            # 5. Save
            df.to_csv(output_names[sheet], index=False)
            print(f"Created: {output_names[sheet]}")

        except Exception as e:
            print(f"Error on {sheet}: {e}")

# Run it
extract_simple_tables(file_path)

Created: SAL_Decile_Disadvantage.csv
Created: SAL_Decile_Adv_Disadv.csv
Created: SAL_Decile_Economic_Resources.csv
Created: SAL_Decile_Employment.csv


In [5]:
import pandas as pd

# The names of the 4 files you created in the previous step
files = [
    'SAL_Decile_Disadvantage.csv',
    'SAL_Decile_Adv_Disadv.csv',
    'SAL_Decile_Economic_Resources.csv',
    'SAL_Decile_Employment.csv'
]

def filter_by_sal_range():
    for file_name in files:
        try:
            # Load the CSV
            df = pd.read_csv(file_name)

            # 1. Convert SAL Code to numeric to allow range filtering
            # errors='coerce' turns non-numeric junk into NaN
            df['SAL Code'] = pd.to_numeric(df['SAL Code'], errors='coerce')

            # 2. Filter: Keep only rows where SAL Code is between 40001 and 49999
            mask = (df['SAL Code'] >= 40001) & (df['SAL Code'] <= 49999)
            df_filtered = df[mask].copy()

            # 3. Convert SAL Code back to integer (removes the .0)
            df_filtered['SAL Code'] = df_filtered['SAL Code'].astype(int)

            # 4. Overwrite the file or save as a new filtered version
            output_name = f"Filtered_{file_name}"
            df_filtered.to_csv(output_name, index=False)

            print(f"Finished {file_name}: Kept {len(df_filtered)} rows.")

        except Exception as e:
            print(f"Error filtering {file_name}: {e}")

if __name__ == "__main__":
    filter_by_sal_range()

Finished SAL_Decile_Disadvantage.csv: Kept 1698 rows.
Finished SAL_Decile_Adv_Disadv.csv: Kept 1698 rows.
Finished SAL_Decile_Economic_Resources.csv: Kept 1698 rows.
Finished SAL_Decile_Employment.csv: Kept 1698 rows.


In [6]:
import pandas as pd

# Path to your file in Colab
file_path = '/content/Suburbs and Localities, SA1 Distributions, SEIFA 2021.xlsx'

# mapping of sheets to your requested column names
# Table 1: Disadvantage
# Table 2: Advantage & Disadvantage
# Table 3: Economic Resources
# Table 4: Employment
sheet_map = {
    'Table 1': 'sal_dis',
    'Table 2': 'sal_adv_dis',
    'Table 3': 'sal_Economic_Resources',
    'Table 4': 'sal_Employment'
}

# List to hold the 4 dataframes
dfs = []

for sheet, new_col_name in sheet_map.items():
    # 1. Read first 3 columns and skip the ABS header metadata
    df = pd.read_excel(file_path, sheet_name=sheet, skiprows=5, usecols=[0, 1, 2])

    # 2. Rename columns to standard names and your specific rank name
    df.columns = ['salcode', 'salname', new_col_name]

    # 3. Filter for SAL Codes between 40001 and 49999 (South Australia)
    df['salcode'] = pd.to_numeric(df['salcode'], errors='coerce')
    df = df.dropna(subset=['salcode'])
    df = df[(df['salcode'] >= 40001) & (df['salcode'] <= 49999)]

    # 4. Clean up formatting
    df['salcode'] = df['salcode'].astype(int)
    df['salname'] = df['salname'].astype(str).str.strip()

    dfs.append(df)

# Merge all 4 tables on salcode and salname
# We use an 'outer' merge just in case a suburb exists in one table but not another
final_df = dfs[0]
for next_df in dfs[1:]:
    final_df = pd.merge(final_df, next_df, on=['salcode', 'salname'], how='outer')

# Reorder columns exactly as you requested
column_order = ['salcode', 'salname', 'sal_adv_dis', 'sal_dis', 'sal_Economic_Resources', 'sal_Employment']
final_df = final_df[column_order]

# Save the final combined file
final_output = 'Combined_SEIFA_2021_SA.csv'
final_df.to_csv(final_output, index=False)

print(f"Success! Created {final_output} with {len(final_df)} rows.")

Success! Created Combined_SEIFA_2021_SA.csv with 1698 rows.


In [7]:
# Load dataset
df = pd.read_csv("/content/Combined_SEIFA_2021_SA.csv")
# 1. Missing values
print("Missing values:\n", df.isnull().sum())
print("\n")

# 3. Unique values per column
print("Unique values:\n", df.nunique())
print("\n")
print(df.shape)



Missing values:
 salcode                    0
salname                    0
sal_adv_dis               85
sal_dis                   85
sal_Economic_Resources    85
sal_Employment            84
dtype: int64


Unique values:
 salcode                   1698
salname                   1698
sal_adv_dis                 10
sal_dis                     10
sal_Economic_Resources      10
sal_Employment              10
dtype: int64


(1698, 6)


In [8]:
# Load dataset
df = pd.read_csv("/content/data_sa_crime_q1_q2_q3_q4_2024-25-publish.csv") #https://data.sa.gov.au/data/dataset/crime-statistics/resource/4a553fc6-71fe-4dac-8096-c29abc269c76
# 1. Missing values
print("Missing values:\n", df.isnull().sum())
print("\n")

# 3. Unique values per column
print("Unique values:\n", df.nunique())
print("\n")
print(df.shape)

Missing values:
 Reported Date                    0
Suburb - Incident              309
Postcode - Incident            331
Offence Level 1 Description      0
Offence Level 2 Description      0
Offence Level 3 Description      0
Offence count                    0
dtype: int64


Unique values:
 Reported Date                   365
Suburb - Incident              1249
Postcode - Incident             382
Offence Level 1 Description       2
Offence Level 2 Description      11
Offence Level 3 Description      31
Offence count                    17
dtype: int64


(95703, 7)


In [9]:
import pandas as pd

# --- 1. SET FILE PATHS ---
# Ensure these files are uploaded to the 'content' folder in your Colab sidebar
excel_file = '/content/Suburbs and Localities, SA1 Distributions, SEIFA 2021.xlsx'
crime_file = '/content/data_sa_crime_q1_q2_q3_q4_2024-25-publish.csv'

# --- 2. EXTRACT AND CLEAN SEIFA TABLES ---
# Mapping sheets to your requested column names
sheet_map = {
    'Table 1': 'sal_dis',
    'Table 2': 'sal_adv_dis',
    'Table 3': 'sal_Economic_Resources',
    'Table 4': 'sal_Employment'
}

seifa_dfs = []

print("Processing SEIFA Tables...")
for sheet, rank_col in sheet_map.items():
    # Read first 3 columns (Code, Name, Rank), skip ABS metadata header (5 rows)
    df = pd.read_excel(excel_file, sheet_name=sheet, skiprows=5, usecols=[0, 1, 2])

    # Standardize column names
    df.columns = ['salcode', 'salname', rank_col]

    # Filter for South Australia SAL Codes (40001 to 49999)
    df['salcode'] = pd.to_numeric(df['salcode'], errors='coerce')
    df = df.dropna(subset=['salcode'])
    df = df[(df['salcode'] >= 40001) & (df['salcode'] <= 49999)]

    # Formatting cleanup
    df['salcode'] = df['salcode'].astype(int)
    df['salname'] = df['salname'].astype(str).str.strip()

    seifa_dfs.append(df)

# --- 3. MERGE SEIFA TABLES INTO MASTER LIST ---
# Merge all 4 on salcode and salname
combined_seifa = seifa_dfs[0]
for next_df in seifa_dfs[1:]:
    combined_seifa = pd.merge(combined_seifa, next_df, on=['salcode', 'salname'], how='outer')

# Reorder columns as requested
final_cols = ['salcode', 'salname', 'sal_adv_dis', 'sal_dis', 'sal_Economic_Resources', 'sal_Employment']
combined_seifa = combined_seifa[final_cols]

# Save intermediate master SEIFA file
combined_seifa.to_csv('Combined_SEIFA_2021_SA.csv', index=False)
print("Created: Combined_SEIFA_2021_SA.csv")

# --- 4. JOIN WITH CRIME DATA ---
print("Joining with Crime Data...")
crime_df = pd.read_csv(crime_file)

# Normalize names for matching (handle Case Sensitivity and extra spaces)
combined_seifa['match_key'] = combined_seifa['salname'].str.upper().str.strip()
crime_df['match_key'] = crime_df['Suburb - Incident'].astype(str).str.upper().str.strip()

# Inner join
final_joined_data = pd.merge(crime_df, combined_seifa, on='match_key', how='inner')

# Drop the helper join column
final_joined_data = final_joined_data.drop(columns=['match_key'])

# --- 5. EXPORT FINAL RESULT ---
output_filename = 'SA_Crime_Joined_With_SEIFA_2021.csv'
final_joined_data.to_csv(output_filename, index=False)

print("-" * 30)
print(f"DONE! Final file saved as: {output_filename}")
print(f"Total rows in final dataset: {len(final_joined_data)}")
print("-" * 30)

# Show a preview
final_joined_data.head()

Processing SEIFA Tables...
Created: Combined_SEIFA_2021_SA.csv
Joining with Crime Data...
------------------------------
DONE! Final file saved as: SA_Crime_Joined_With_SEIFA_2021.csv
Total rows in final dataset: 76657
------------------------------


,Reported Date,Suburb - Incident,Postcode - Incident,Offence Level 1 Description,Offence Level 2 Description,Offence Level 3 Description,Offence count,salcode,salname,sal_adv_dis,sal_dis,sal_Economic_Resources,sal_Employment
0,01/07/2024,ADELAIDE,5000,OFFENCES AGAINST PROPERTY,PROPERTY DAMAGE AND ENVIRONMENTAL,Other property damage and environmental,3,40002,Adelaide,9.0,4.0,1.0,10.0
1,01/07/2024,ADELAIDE,5000,OFFENCES AGAINST PROPERTY,SERIOUS CRIMINAL TRESPASS,SCT - Non Residence,1,40002,Adelaide,9.0,4.0,1.0,10.0
2,01/07/2024,ADELAIDE,5000,OFFENCES AGAINST PROPERTY,THEFT AND RELATED OFFENCES,Other theft,2,40002,Adelaide,9.0,4.0,1.0,10.0
3,01/07/2024,ADELAIDE,5000,OFFENCES AGAINST PROPERTY,THEFT AND RELATED OFFENCES,Theft from motor vehicle,1,40002,Adelaide,9.0,4.0,1.0,10.0
4,01/07/2024,ADELAIDE,5000,OFFENCES AGAINST PROPERTY,THEFT AND RELATED OFFENCES,Theft from shop,4,40002,Adelaide,9.0,4.0,1.0,10.0


In [10]:
# Load dataset
df = pd.read_csv("/content/SA_Crime_Joined_With_SEIFA_2021.csv")
# 1. Missing values
print("Missing values:\n", df.isnull().sum())
print("\n")

# 3. Unique values per column
print("Unique values:\n", df.nunique())
print("\n")
print(df.shape)

Missing values:
 Reported Date                    0
Suburb - Incident                0
Postcode - Incident              0
Offence Level 1 Description      0
Offence Level 2 Description      0
Offence Level 3 Description      0
Offence count                    0
salcode                          0
salname                          0
sal_adv_dis                    875
sal_dis                        875
sal_Economic_Resources         875
sal_Employment                 746
dtype: int64


Unique values:
 Reported Date                  365
Suburb - Incident              965
Postcode - Incident            312
Offence Level 1 Description      2
Offence Level 2 Description     10
Offence Level 3 Description     28
Offence count                   17
salcode                        965
salname                        965
sal_adv_dis                     10
sal_dis                         10
sal_Economic_Resources          10
sal_Employment                  10
dtype: int64


(76657, 13)


In [11]:
import pandas as pd
import numpy as np

# Load the joined dataset
df = pd.read_csv('SA_Crime_Joined_With_SEIFA_2021.csv')

# Ensure Date is in datetime format
df['Reported Date'] = pd.to_datetime(df['Reported Date'], dayfirst=True)

# 1. Basic Time Features
df['Cal_Month'] = df['Reported Date'].dt.month
df['Cal_DayOfMonth'] = df['Reported Date'].dt.day
df['Cal_DayOfWeek'] = df['Reported Date'].dt.day_name()
df['IsWeekend'] = df['Reported Date'].dt.dayofweek.isin([5, 6]).astype(int)

# 2. Season (Southern Hemisphere)
def get_season(month):
    if month in [12, 1, 2]: return 'Summer'
    if month in [3, 4, 5]: return 'Autumn'
    if month in [6, 7, 8]: return 'Winter'
    return 'Spring'

df['Season_SH'] = df['Cal_Month'].apply(get_season)

# 3. Public Holidays (South Australia 2024-2025)
sa_public_holidays = pd.to_datetime([
    '2024-01-01', '2024-01-26', '2024-03-11', '2024-03-29', '2024-03-30', '2024-04-01',
    '2024-04-25', '2024-06-10', '2024-10-07', '2024-12-25', '2024-12-26',
    '2025-01-01', '2025-01-27', '2025-03-10', '2025-04-18', '2025-04-19', '2025-04-20',
    '2025-04-21', '2025-04-25', '2025-06-09', '2025-10-06', '2025-12-25', '2025-12-26'
])
df['IsPublicHoliday_SA'] = df['Reported Date'].isin(sa_public_holidays).astype(int)

# 4. Long Weekend (Holiday adjacent to a weekend)
# Defined as a Friday or Monday that is a public holiday
df['IsLongWeekend'] = (
    (df['IsPublicHoliday_SA'] == 1) &
    (df['Reported Date'].dt.dayofweek.isin([0, 4]))
).astype(int)

# 5. School Holidays (South Australia 2024-2025)
def is_sa_school_holiday(date):
    # Defining SA School Holiday Ranges
    ranges = [
        ('2024-04-13', '2024-04-28'), # Autumn 24
        ('2024-07-06', '2024-07-21'), # Winter 24
        ('2024-09-28', '2024-10-13'), # Spring 24
        ('2024-12-14', '2025-01-26'), # Summer 24/25
        ('2025-04-12', '2025-04-27'), # Autumn 25
        ('2025-07-05', '2025-07-20'), # Winter 25
        ('2025-09-27', '2025-10-12'), # Spring 25
        ('2025-12-13', '2025-12-31')  # Summer 25 start
    ]
    for start, end in ranges:
        if pd.Timestamp(start) <= date <= pd.Timestamp(end):
            return 1
    return 0

df['IsSchoolHoliday_SA'] = df['Reported Date'].apply(is_sa_school_holiday)

# --- Save New Dataset ---
df.to_csv('Complex_SA_Crime_Features.csv', index=False)

print(f"Features Generated. New Column Count: {len(df.columns)}")
print(df[['Reported Date', 'Season_SH', 'IsPublicHoliday_SA', 'IsSchoolHoliday_SA']].head())

Features Generated. New Column Count: 21
  Reported Date Season_SH  IsPublicHoliday_SA  IsSchoolHoliday_SA
0    2024-07-01    Winter                   0                   0
1    2024-07-01    Winter                   0                   0
2    2024-07-01    Winter                   0                   0
3    2024-07-01    Winter                   0                   0
4    2024-07-01    Winter                   0                   0


In [12]:
# Load dataset
df = pd.read_csv("/content/Complex_SA_Crime_Features.csv")
# 1. Missing values
print("Missing values:\n", df.isnull().sum())
print("\n")

# 3. Unique values per column
print("Unique values:\n", df.nunique())
print("\n")
print(df.shape)

Missing values:
 Reported Date                    0
Suburb - Incident                0
Postcode - Incident              0
Offence Level 1 Description      0
Offence Level 2 Description      0
Offence Level 3 Description      0
Offence count                    0
salcode                          0
salname                          0
sal_adv_dis                    875
sal_dis                        875
sal_Economic_Resources         875
sal_Employment                 746
Cal_Month                        0
Cal_DayOfMonth                   0
Cal_DayOfWeek                    0
IsWeekend                        0
Season_SH                        0
IsPublicHoliday_SA               0
IsLongWeekend                    0
IsSchoolHoliday_SA               0
dtype: int64


Unique values:
 Reported Date                  365
Suburb - Incident              965
Postcode - Incident            312
Offence Level 1 Description      2
Offence Level 2 Description     10
Offence Level 3 Description     28
Offence

To check for rebundancy of data

In [13]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Load your current dataset
df = pd.read_csv('/content/Complex_SA_Crime_Features.csv')

# --- 1. CLEANING ---
# Remove exact duplicates to ensure data quality
df_clean = df.drop_duplicates()

# --- 2. SELECTING SPECIFIC COLUMNS ---
# We keep 'salcode' and 'Postcode - Incident'
# We remove 'salname' because it is identical to 'Suburb - Incident'
cols_to_drop = ['salname']
df_clean = df_clean.drop(columns=cols_to_drop, errors='ignore')

# --- 3. FINAL VERIFICATION ---
print(f"Original Rows: {len(df)} | Cleaned Rows: {len(df_clean)}")
print(f"Current Column Count: {len(df_clean.columns)}")
print("\nFirst 5 rows of your final spatial mapping:")
print(df_clean[['salcode', 'Suburb - Incident', 'Postcode - Incident']].head())

# --- 4. SAVE AND DOWNLOAD ---
output_file = 'Final_Spatial_Crime_SEIFA.csv'
df_clean.to_csv(output_file, index=False)

# Optional: Trigger download in Colab
# from google.colab import files
# files.download(output_file)

Original Rows: 76657 | Cleaned Rows: 76598
Current Column Count: 20

First 5 rows of your final spatial mapping:
   salcode Suburb - Incident  Postcode - Incident
0    40002          ADELAIDE                 5000
1    40002          ADELAIDE                 5000
2    40002          ADELAIDE                 5000
3    40002          ADELAIDE                 5000
4    40002          ADELAIDE                 5000


In [15]:
df = pd.read_csv("/content/Final_Spatial_Crime_SEIFA.csv")
print(df.shape)


(76598, 20)
